In [ ]:
import pandas as pd
from load import MergedExposureDataset, MonitorsDataset, ExposureDataset, SmokePMDataset
import matplotlib.pyplot as plt
import geopandas as gpd
import textwrap
from config import *

sdf = gpd.read_file(STATESHAPEFILE_PATH)
sdf = sdf[sdf['STUSPS'].isin(['AZ', 'CA', 'CO', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA','WY'])]

In [ ]:
MonitorsDataset().plotMonitors()

In [ ]:
df = MergedExposureDataset(reload=True).getMergedExposure()

In [ ]:
df.columns

# Person-Days Calculations

In [ ]:
df['smoke_person_days'] = df['POPULATION'] * df['sum_smoke_pm_gt_5']
df['pm25_person_days'] = df['POPULATION'] * df['sum_pm25_gt_35']

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=df.geometry, crs="ESRI:102003")

In [ ]:
sdf = sdf.to_crs(gdf.crs)

In [ ]:
gdf["POP_SQMI"] = gdf["POPULATION"] / gdf["SQMI"]

In [ ]:
gdf["wg_exp_d"] = gdf["popw_avg_yrly_smoke_pm"] * gdf["distance"]

In [ ]:
def draw_popw(ax, annotation):
    norm = None
    #fig, ax = plt.subplots(1, 1, figsize=(3, 3))

    #gdf["wg_exp"] = gdf["popw_avg_yrly_smoke_pm"] * gdf["POPULATION"]
    x = "popw_avg_yrly_smoke_pm"
    gdf.plot(
        column=x, 
        ax=ax,
        legend=True, 
        linewidth=0, 
        cmap="viridis_r", 
        missing_kwds={"color": "lightgrey"}, norm = norm
        )
    sdf.boundary.plot(ax=ax, linewidth=0.3, color="white")
    wrapped_title = textwrap.fill(
        r"Average population-weighted wildfire smoke $PM_{2.5}$", 40)
    ax.set_title(wrapped_title, fontsize=10)
    ax.set_axis_off()

    plt.tight_layout()
    ax.annotate(annotation, xy=(0.02, 0.02), xycoords="axes fraction", fontsize=12, fontweight="bold")
    #plt.savefig("figures/wg_exp.png", dpi=300, bbox_inches='tight')


def plotMonitors(ax, annotation=None):
    df = MonitorsDataset().loadMonitors()

    sdf = gpd.read_file(STATESHAPEFILE_PATH)
    sdf = sdf[sdf['STUSPS'].isin(['AZ', 'CA', 'CO', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA','WY'])]

    x = "POP_SQMI"
    gdf.plot(
        column=x, 
        ax=ax,
        legend=True, 
        linewidth=0, 
        cmap="viridis_r", 
        missing_kwds={"color": "lightgrey"}, 
        )

    # gdf.plot(
    #     ax=ax, 
    #     color='white',
    #     edgecolor='lightgrey',
    #     legend=True, 
    #     vmax=200,
    #     linewidth=0.5, 
    #     missing_kwds={"color": "lightgrey"},
    #     )

    #df[df["State Code"] != 48].plot(ax=ax, color='darkblue', marker='^', markersize=10)
    
    sdf = sdf.to_crs(gdf.crs)

    #sdf.boundary.plot(ax=ax, linewidth=0.3, color="grey")
    title_text = r"Active $PM_{2.5}$ monitoring site locations in 2024"
    wrapped_title = textwrap.fill(title_text, 40)
    ax.set_title(wrapped_title, fontsize=10)

    ax.set_axis_off()
    #ax.annotate(annotation, xy=(0.02, 0.02), xycoords="axes fraction", fontsize=12, fontweight="bold")
    plt.tight_layout()


In [ ]:
def add_figure(ax, annotation, x=None, title=None, cmap="viridis_r", vmax=None):
    if vmax is None:
        vmax = gdf[x].quantile(0.99)
    else:
        vmax = gdf[x].quantile(vmax*0.01)
    norm = plt.Normalize(vmin=0, vmax=vmax)  
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    gdf.plot(
        column=x, 
        ax=ax,
        vmax=vmax,
        legend=True, 
        linewidth=0, 
        cmap=cmap, 
        legend_kwds={'shrink': 0.5},
        missing_kwds={"color": "lightgrey"}, norm = norm
        )
    sdf.boundary.plot(ax=ax, linewidth=0.3, color="white")
    wrapped_title = textwrap.fill(title, 45) 
    
    # Add colorbar manually (much cleaner)
    # plt.colorbar(sm, ax=ax, shrink=0.8)   # <--- shrink legend to 80%

    ax.set_title(wrapped_title, fontsize=10)
    ax.set_axis_off()
    ax.annotate(annotation, xy=(0.02, 0.02), xycoords="axes fraction", fontsize=12, fontweight="bold")
    # plt.tight_layout()
    # plt.savefig("figures/wg_exp_d.png", dpi=300, bbox_inches='tight')
    # plt.show()

In [ ]:
days_cmap = "magma_r"
conc_cmap = "inferno_r"

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(8.26, 9.5))

add_figure(
    axes[0, 0],
    "(A)",
    x="avg_yrly_pm25_gt_35",
    title="Average number of high air pollution days ($PM_{2.5}>35μg/m^3$)",
    cmap="magma_r",
)
add_figure(
    axes[0, 1],
    "(B)",
    x="avg_yrly_smoke_pm_gt_5",
    title="Average number of wildfire smoke days ($PM_{2.5}>5μg/m^3$)",
    cmap="magma_r",
)

add_figure(
    axes[0, 2],
    "(C)",
    x="avg_yrly_pm25",
    title="Average annual mean concentration of total ambient $PM_{2.5}$ ($μg/m^3$)",
    cmap=conc_cmap,
)
add_figure(
    axes[1, 0],
    "(D)",
    x="avg_yrly_smoke_pm",
    title="Average annual mean concentration of wildfire smoke $PM_{2.5}$ ($μg/m^3$)",
    cmap=conc_cmap,
)

add_figure(
    axes[1, 1],
    "(E)",
    x="pm25_person_days",
    title=r"Total person-days of exposure to high air pollution (days with $PM_{2.5}>35μg/m^3$)",
)
add_figure(
    axes[1, 2],
    "(F)",
    x="smoke_person_days",
    title=r"Total person-days of exposure to wildfire smoke (days with $PM_{2.5}>5μg/m^3$)",
)

add_figure(
    axes[2, 0],
    "(G)",
    x="wg_exp_d",
    title=r"Monitor-proximity and population-weighted wildfire smoke exposure",
)
add_figure(
    axes[2, 1],
    "(H)",
    x="POP_SQMI",
    title=r"Population density by census tract (people $mi^{2}$)",
    vmax=80,
)

MonitorsDataset().plotMonitors(ax=axes[2, 2], annotation="(I)")

plt.tight_layout()
plt.show()
# add_figure(axes[0, 1], "(B)", x="avg_yrly_pm25_gt_5", title="Average number of air pollution days ($PM_{2.5}>5μg/m^3$)", cmap="magma_r")

In [ ]:
# Number of people living in rural Washington state

df[(df['STATE_ABBR']=='CA') & (df['classification']=='Rural')]['POPULATION'].sum()

# Total WA population 7680786.0
# Rural WA population 1180772.0

# Total CA population 39507131.0
# Rural CA population 1833043.0

In [ ]:
gdf["POPULATION"] = gdf["POPULATION"].fillna(0).astype(int)

In [ ]:
gdf['wg_exp_d'] = gdf['wg_exp_d'].round(5)

In [ ]:
gdf.sort_values("wg_exp_d", ascending=False)[["classification","STATE_ABBR","POPULATION","wg_exp_d"]].head(10)

In [ ]:
temp = gpd.read_file("../coexposure/data/WesternUSCensusTract/CensusTract2020_WesternUS.shp")

In [ ]:
temp["county_fips"] = temp["FIPS"].str[0:5]

In [ ]:
temp.county_fips.nunique()

In [ ]:
gdf["avg_yrly_smoke_pm_gt_5_person_days"] = \
    gdf["avg_yrly_smoke_pm_gt_5"] * gdf["POPULATION"]